# Stage 2 — Phase Recognition (Full Pipeline)

Assigns a surgical phase to every sampled frame of a distal hypospadias repair
video. This notebook is self-contained for Stage 2: it builds the manifest,
extracts frozen-backbone features, trains a BiGRU phase-recognition model
under leave-one-video-out cross-validation, and reports the pooled metrics,
segmental metrics, and the pre-specified Model C vs Model D statistical
comparison.

**Everything fitted to data (model weights, feature scalers, class weights)
is rebuilt inside each fold using only that fold's six training videos.**
Nothing that touches all seven videos is used at test time — that would leak
the held-out video into its own evaluation.

Run cells top to bottom. `CONFIG` below is the only place you should need to
edit paths before running against real data.

## Execution order
1. Manifest, per-frame labels, incidence table and frame counts
2. Feature extraction and caching, all four backbones (+ detector features for Model D)
3. t-SNE per backbone coloured by phase (sanity check, feeds Stage 3)
4. Majority-class baseline
5. Per-frame baseline
6. Backbone sweep with GRU
7. Model C vs Model D on the winning backbone


## 0. Colab setup

Mounts Google Drive (source of truth for videos, CVAT annotations, and all
cached artefacts — nothing here should live only in the ephemeral Colab VM)
and installs the handful of packages Colab doesn't ship by default.


In [ ]:
# If not running in Colab this is a no-op fallback so the notebook still
# works locally against a filesystem path instead of Drive.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not running in Colab — set CONFIG['project_root'] to a local path below.")


In [ ]:
%pip install -q opencv-python-headless einops scikit-learn scipy pandas seaborn tqdm


In [ ]:
import os, io, json, math, time, random, warnings, subprocess
from pathlib import Path
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from sklearn.metrics import f1_score, confusion_matrix, classification_report
from scipy.stats import wilcoxon
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=UserWarning)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


## 1. Global configuration

- `project_root`: everything reads/writes under here (Drive path in Colab).
- `videos`: the seven in-house video IDs. EgoSurgery-Phase is **not** used —
  mapping its nine generic open-surgery phases onto this fine-grained
  hypospadias taxonomy would need a subjective single-rater correspondence
  too loose to defend, and at n=7 a transfer arm wouldn't yield an
  interpretable result. It is intentionally absent, not forgotten.
- `phase_taxonomy`: edit to match your CVAT label set exactly. Order fixes
  the class index used everywhere below (confusion matrices, class weights).
- `gap_policy`: frames in gaps between annotated intervals are either
  `"background"` (kept as their own class) or `"drop"`ped. This is a single
  choice, applied consistently, and it changes what macro-F1 measures — so
  it is recorded in every run's metadata (Section 11), not just chosen once
  and forgotten.


In [ ]:
@dataclass
class Config:
    project_root: str = "/content/drive/MyDrive/hypospadias_stage2"

    # Seven in-house distal hypospadias videos, from Timestamps.xlsx.
    # IDs are the bare video numbers, matching the raw filenames (1.mp4,
    # 2.mp4, ...) rather than a "V1"-style prefix. Note video 3 is
    # intentionally absent — the source spreadsheet's "Video #" column
    # skips from 2 to 4, so this cohort really is {1,2,4,5,6,7,8}.
    # raw_videos/ must contain 1.mp4, 2.mp4, 4.mp4, 5.mp4, 6.mp4, 7.mp4,
    # 8.mp4 (no 3.mp4) for these IDs to resolve.
    videos: list = field(default_factory=lambda: [
        "1", "2", "4", "5", "6", "7", "8",
    ])

    # Fine-grained hypospadias phase taxonomy — the 22 distinct "Phase
    # Label" values found in Timestamps.xlsx, plus "background" for gaps
    # between annotated intervals. Order fixes the class index used
    # everywhere below (confusion matrices, class weights); grouped here
    # in roughly the order phases occur within a repair, purely for
    # readability. If CVAT labels ever diverge from these exact strings
    # (case-sensitive), build_full_manifest raises rather than mislabels.
    phase_taxonomy: list = field(default_factory=lambda: [
        "background",
        "Preoperative anatomy",
        "Skin marking",
        "Epinephrine",
        "Degloving",
        "Artificial erection test",
        "Tourniquet applied",
        "Orthoplasty",
        "Glans marking",
        "Glans incision",
        "Skin mobilisation",
        "Glans wings mobilisation",
        "Urethral plate incision",
        "Tourniquet released",
        "Urethroplasty",
        "Barrier layer coverage",
        "Glansplasty",
        "Foreskin reconstruction",
        "Circumcision",
        "Skin edge closure",
        "Skin closure",
        "End of operation anatomy",
        "Dressing",
    ])

    gap_policy: str = "background"  # "background" or "drop"

    base_fps: float = 2.0
    min_frames_per_segment: int = 10

    window_size: int = 1024
    window_stride: int = 512

    seed: int = 42
    n_folds: int = 7  # == len(videos), leave-one-video-out

    # Detector classes used for Model D (Bovie excluded — 8 detections
    # across 140 sampled frames in Stage 1 made it unusable as a feature
    # and unevaluable as a class).
    detector_classes: list = field(default_factory=lambda: [
        "hand", "needle_driver", "forceps",
    ])

    backbones: list = field(default_factory=lambda: [
        "resnet50_supervised", "dino_vitb16", "dinov2_vitb14", "mocov3_resnet50",
    ])

    # Path to a MoCo v3 ResNet-50 checkpoint (not on torch.hub — download
    # from https://github.com/facebookresearch/moco-v3 and point here).
    mocov3_checkpoint: str = "/content/drive/MyDrive/hypospadias_stage2/checkpoints/mocov3_r50_linear-vitb.pth.tar"


CFG = Config()

ROOT = Path(CFG.project_root)
DIRS = {
    "videos": ROOT / "raw_videos",
    "annotations": ROOT / "annotations",          # CVAT exports, one per video
    "manifest": ROOT / "manifest",
    "features": ROOT / "features",                # per-backbone frame feature caches
    "detector": ROOT / "detector_features",        # Stage 1 output, per video
    "tsne": ROOT / "tsne",
    "results": ROOT / "results",
    "logits": ROOT / "results" / "logits",
    "checkpoints": ROOT / "checkpoints",
}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

PHASE_TO_IDX = {p: i for i, p in enumerate(CFG.phase_taxonomy)}
IDX_TO_PHASE = {i: p for p, i in PHASE_TO_IDX.items()}
N_CLASSES = len(CFG.phase_taxonomy)


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(CFG.seed)
print(f"{len(CFG.videos)} videos, {N_CLASSES} phase classes, gap_policy={CFG.gap_policy!r}")


## 2. Manifest, per-frame labels, incidence table and frame counts

### 2.1 Reading phase boundaries out of CVAT

Phase boundaries were annotated in CVAT as a single interpolated track per
phase, with the shape toggled `outside` to mark the interval where that
phase is active. `parse_cvat_intervals` collapses the keyframes of each
track back into `(phase_label, start_frame, end_frame)` intervals. Phases
may recur non-contiguously — in the actual cohort, video 7 contains
`Glans wings mobilisation` twice with a `Tourniquet applied` segment
between, and video 8 contains `Tourniquet applied`, `Barrier layer
coverage`, and `Tourniquet released` each twice — this is legitimate and
just means one label maps to multiple intervals, which is exactly what a
list of intervals (rather than a single start/end column) represents.

If your CVAT export is already a flat interval table, skip straight to
`load_interval_csv` below instead.


In [ ]:
def parse_cvat_intervals(xml_path: str) -> pd.DataFrame:
    """Parse a CVAT-for-video 1.1 XML export into (phase, start_frame, end_frame).

    Expects one <track label="phase_name"> per phase occurrence, with
    <box>/<points> keyframes whose `outside` attribute toggles 0 (visible,
    phase active) / 1 (phase not active). Contiguous outside=0 runs become
    one interval each, so a phase that recurs (e.g. two separate tracks, or
    one track with two visible runs) naturally produces multiple intervals.
    """
    import xml.etree.ElementTree as ET

    tree = ET.parse(xml_path)
    root = tree.getroot()
    rows = []
    for track in root.findall(".//track"):
        label = track.get("label")
        shapes = sorted(
            list(track.findall("box")) + list(track.findall("points")),
            key=lambda s: int(s.get("frame")),
        )
        run_start = None
        prev_frame = None
        for shape in shapes:
            frame = int(shape.get("frame"))
            outside = shape.get("outside", "0") == "1"
            if not outside and run_start is None:
                run_start = frame
            if outside and run_start is not None:
                rows.append((label, run_start, prev_frame))
                run_start = None
            prev_frame = frame
        if run_start is not None:
            rows.append((label, run_start, prev_frame))
    return pd.DataFrame(rows, columns=["phase", "start_frame", "end_frame"])


def load_interval_csv(csv_path: str) -> pd.DataFrame:
    """Flat interval export: columns phase, start_frame, end_frame (or
    start_sec/end_sec, converted below once fps is known)."""
    return pd.read_csv(csv_path)


def load_video_intervals(video_id: str, fps: float) -> pd.DataFrame:
    xml_path = DIRS["annotations"] / f"{video_id}.xml"
    csv_path = DIRS["annotations"] / f"{video_id}.csv"
    if xml_path.exists():
        df = parse_cvat_intervals(str(xml_path))
    elif csv_path.exists():
        df = load_interval_csv(str(csv_path))
        if "start_frame" not in df.columns and "start_sec" in df.columns:
            df["start_frame"] = (df["start_sec"] * fps).round().astype(int)
            df["end_frame"] = (df["end_sec"] * fps).round().astype(int)
    else:
        raise FileNotFoundError(
            f"No CVAT export found for {video_id} at {xml_path} or {csv_path}"
        )
    df["video_id"] = video_id
    df["start_time"] = df["start_frame"] / fps
    df["end_time"] = df["end_frame"] / fps
    return df[["video_id", "phase", "start_frame", "end_frame", "start_time", "end_time"]]


### 2.2 No frame may receive two labels

Overlapping intervals within a video (regardless of phase) would mean some
frame has two candidate labels, which is a contradiction the manifest must
never silently resolve. `assert_no_overlap` fails loudly and names the
offending pair instead.


In [ ]:
def assert_no_overlap(intervals: pd.DataFrame, video_id: str):
    ivs = intervals.sort_values("start_frame").reset_index(drop=True)
    for i in range(len(ivs) - 1):
        a, b = ivs.iloc[i], ivs.iloc[i + 1]
        if b["start_frame"] < a["end_frame"]:
            raise AssertionError(
                f"[{video_id}] overlapping intervals: "
                f"{a['phase']}[{a['start_frame']}:{a['end_frame']}] overlaps "
                f"{b['phase']}[{b['start_frame']}:{b['end_frame']}]"
            )


### 2.3 Sampling: base rate + dense resampling of short segments

Base rate is 2 fps across the whole video. Any segment (a single contiguous
interval) whose base-rate sample count falls below
`CFG.min_frames_per_segment` (10) is densely, evenly resampled up to that
floor — this exists because brief event phases (`tourniquet_released` spans
1–8 seconds across the cohort) would otherwise contribute too few frames to
be learnable at all. The `sampling_mode` column records which frames are
base-rate and which are the dense top-up, so a strong result on a rare phase
can later be checked against "was this just near-duplicate frames."


In [ ]:
def get_video_fps_and_length(video_path: str):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    return fps, n_frames


def sample_segment(start_frame, end_frame, native_fps, base_fps, min_frames):
    """Return (frame_idx, sampling_mode) pairs for one contiguous interval."""
    step = max(1, round(native_fps / base_fps))
    base_frames = list(range(start_frame, end_frame + 1, step))
    if len(base_frames) == 0:
        base_frames = [start_frame]
    rows = [(f, "base") for f in base_frames]

    if len(base_frames) < min_frames and end_frame > start_frame:
        dense_frames = np.linspace(start_frame, end_frame, num=min_frames)
        dense_frames = sorted(set(int(round(f)) for f in dense_frames))
        existing = set(base_frames)
        rows += [(f, "dense") for f in dense_frames if f not in existing]
    return rows


def build_video_manifest(video_id: str) -> pd.DataFrame:
    video_path = str(DIRS["videos"] / f"{video_id}.mp4")
    native_fps, n_frames = get_video_fps_and_length(video_path)

    intervals = load_video_intervals(video_id, native_fps)
    assert_no_overlap(intervals, video_id)

    rows = []
    for _, iv in intervals.iterrows():
        for frame_idx, mode in sample_segment(
            int(iv.start_frame), int(iv.end_frame), native_fps, CFG.base_fps, CFG.min_frames_per_segment
        ):
            rows.append((video_id, frame_idx, frame_idx / native_fps, iv.phase, mode))

    # Gaps between annotated intervals.
    covered = np.zeros(n_frames, dtype=bool)
    for _, iv in intervals.iterrows():
        covered[int(iv.start_frame): int(iv.end_frame) + 1] = True
    gap_step = max(1, round(native_fps / CFG.base_fps))
    for f in range(0, n_frames, gap_step):
        if not covered[f]:
            if CFG.gap_policy == "background":
                rows.append((video_id, f, f / native_fps, "background", "base"))
            # "drop": simply omit the frame from the manifest entirely.

    df = pd.DataFrame(rows, columns=["video_id", "frame_idx", "timestamp", "phase", "sampling_mode"])
    df = df.drop_duplicates(subset=["video_id", "frame_idx"]).sort_values("frame_idx").reset_index(drop=True)
    return df


def build_full_manifest(save=True) -> pd.DataFrame:
    parts = [build_video_manifest(vid) for vid in tqdm(CFG.videos, desc="manifest")]
    manifest = pd.concat(parts, ignore_index=True)
    unknown = set(manifest.phase) - set(CFG.phase_taxonomy)
    if unknown:
        raise ValueError(f"Manifest contains phases not in CFG.phase_taxonomy: {unknown}")
    manifest["phase_idx"] = manifest.phase.map(PHASE_TO_IDX)
    if save:
        out = DIRS["manifest"] / "frames_manifest.csv"
        manifest.to_csv(out, index=False)
        print(f"Saved manifest: {out} ({len(manifest)} frames)")
    return manifest


# manifest = build_full_manifest()
# manifest.head()


### 2.4 Pre-modelling artefacts: incidence matrix and frame counts

Documents what the dataset can support, independent of any model result —
these belong in the results section regardless of downstream performance.


In [ ]:
def build_incidence_artefacts(manifest: pd.DataFrame):
    incidence = pd.crosstab(manifest.video_id, manifest.phase).reindex(columns=CFG.phase_taxonomy, fill_value=0)
    incidence.to_csv(DIRS["manifest"] / "phase_by_video_incidence.csv")

    frame_counts = manifest.phase.value_counts().reindex(CFG.phase_taxonomy, fill_value=0)
    frame_counts.name = "n_frames"
    frame_counts.to_csv(DIRS["manifest"] / "phase_frame_counts.csv")

    mode_counts = manifest.groupby(["phase", "sampling_mode"]).size().unstack(fill_value=0)
    mode_counts.to_csv(DIRS["manifest"] / "phase_sampling_mode_counts.csv")

    print("Phase x video incidence (frame counts):")
    display(incidence)
    print("\nTotal frames per phase:")
    display(frame_counts)
    print("\nBase-rate vs dense-resampled frames per phase:")
    display(mode_counts)
    return incidence, frame_counts, mode_counts


# incidence, frame_counts, mode_counts = build_incidence_artefacts(manifest)


## 3. Feature extraction and caching (four frozen backbones)

Each backbone is extracted once, cached to disk in frame order matching the
manifest, and reused by every fold and every arm below — extraction happens
outside the CV loop entirely since these are frozen features, not something
fit to training data.

Each backbone gets **its own preprocessing**: patch sizes and normalisation
constants differ across ViT-B/16, ViT-B/14, and the two ResNet-50 variants,
so sharing one transform pipeline would quietly degrade whichever backbone
didn't match it. ~1.4GB cached across all four.

| Backbone | Dim | Pooling |
|---|---|---|
| ResNet-50 (ImageNet-supervised) | 2048 | global average pool |
| DINO v1, ViT-B/16 | 768 | CLS token |
| DINOv2, ViT-B/14 | 768 | CLS token |
| MoCo v3, ResNet-50 | 2048 | global average pool, projection head discarded |


In [ ]:
import torchvision
from torchvision import transforms as T
from torchvision.models import resnet50, ResNet50_Weights

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

BACKBONE_TRANSFORMS = {
    "resnet50_supervised": T.Compose([
        T.ToPILImage(), T.Resize(256), T.CenterCrop(224), T.ToTensor(),
        T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]),
    "dino_vitb16": T.Compose([
        T.ToPILImage(), T.Resize(256), T.CenterCrop(224), T.ToTensor(),
        T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]),
    "dinov2_vitb14": T.Compose([
        # DINOv2 patch size 14: 224 is divisible by 14, matches the released checkpoints.
        T.ToPILImage(), T.Resize(224), T.CenterCrop(224), T.ToTensor(),
        T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]),
    "mocov3_resnet50": T.Compose([
        T.ToPILImage(), T.Resize(256), T.CenterCrop(224), T.ToTensor(),
        T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]),
}

BACKBONE_DIMS = {
    "resnet50_supervised": 2048,
    "dino_vitb16": 768,
    "dinov2_vitb14": 768,
    "mocov3_resnet50": 2048,
}


def load_resnet50_supervised():
    model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
    model.fc = nn.Identity()  # global-average-pooled 2048-d features
    return model.eval().to(DEVICE)


def load_dino_vitb16():
    model = torch.hub.load("facebookresearch/dino:main", "dino_vitb16")
    return model.eval().to(DEVICE)  # forward() returns the CLS token


def load_dinov2_vitb14():
    model = torch.hub.load("facebookresearch/dinov2", "dinov2_vitb14")
    return model.eval().to(DEVICE)  # forward() returns the CLS token


def load_mocov3_resnet50(checkpoint_path: str):
    """MoCo v3 isn't on torch.hub; load the official checkpoint into a plain
    torchvision ResNet-50 and discard the projection/prediction heads,
    keeping only the global-average-pooled encoder trunk (2048-d)."""
    model = resnet50(weights=None)
    model.fc = nn.Identity()
    ckpt = torch.load(checkpoint_path, map_location="cpu")
    state_dict = ckpt.get("state_dict", ckpt)
    new_state = {}
    for k, v in state_dict.items():
        if not k.startswith("module.base_encoder."):
            continue
        nk = k[len("module.base_encoder."):]
        if nk.startswith("fc."):  # projection head, discarded
            continue
        new_state[nk] = v
    missing, unexpected = model.load_state_dict(new_state, strict=False)
    print(f"MoCo v3 load: {len(missing)} missing, {len(unexpected)} unexpected keys "
          "(fc.* unexpected is expected — projection head is discarded)")
    return model.eval().to(DEVICE)


BACKBONE_LOADERS = {
    "resnet50_supervised": load_resnet50_supervised,
    "dino_vitb16": load_dino_vitb16,
    "dinov2_vitb14": load_dinov2_vitb14,
    "mocov3_resnet50": lambda: load_mocov3_resnet50(CFG.mocov3_checkpoint),
}


In [ ]:
class FrameReader:
    """Random-access-ish reader over the specific frame indices a manifest
    needs, without decoding the whole video into memory."""

    def __init__(self, video_path: str):
        self.cap = cv2.VideoCapture(video_path)

    def read_frames(self, frame_indices):
        for idx in frame_indices:
            self.cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ok, frame_bgr = self.cap.read()
            if not ok:
                raise RuntimeError(f"Failed to read frame {idx}")
            yield idx, cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

    def close(self):
        self.cap.release()


@torch.no_grad()
def extract_backbone_features(backbone_name: str, batch_size: int = 32):
    """Extract + cache one backbone's features for every video, in the
    frame order given by that video's manifest rows. Skips videos already
    cached so re-running is cheap."""
    out_dir = DIRS["features"] / backbone_name
    out_dir.mkdir(parents=True, exist_ok=True)
    manifest = pd.read_csv(DIRS["manifest"] / "frames_manifest.csv")
    transform = BACKBONE_TRANSFORMS[backbone_name]
    model = BACKBONE_LOADERS[backbone_name]()

    for video_id in CFG.videos:
        out_path = out_dir / f"{video_id}.npy"
        if out_path.exists():
            continue
        vid_rows = manifest[manifest.video_id == video_id].sort_values("frame_idx")
        frame_indices = vid_rows.frame_idx.tolist()
        reader = FrameReader(str(DIRS["videos"] / f"{video_id}.mp4"))

        feats = np.zeros((len(frame_indices), BACKBONE_DIMS[backbone_name]), dtype=np.float32)
        batch, batch_pos = [], []
        pos = 0
        for _, frame_rgb in tqdm(reader.read_frames(frame_indices), total=len(frame_indices),
                                  desc=f"{backbone_name}/{video_id}"):
            batch.append(transform(frame_rgb))
            batch_pos.append(pos)
            pos += 1
            if len(batch) == batch_size:
                feats[batch_pos] = model(torch.stack(batch).to(DEVICE)).cpu().numpy()
                batch, batch_pos = [], []
        if batch:
            feats[batch_pos] = model(torch.stack(batch).to(DEVICE)).cpu().numpy()
        reader.close()

        np.save(out_path, feats)
    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


def extract_all_backbones():
    for name in CFG.backbones:
        print(f"=== extracting {name} ===")
        extract_backbone_features(name)


def load_cached_features(backbone_name: str, video_id: str) -> np.ndarray:
    return np.load(DIRS["features"] / backbone_name / f"{video_id}.npy")


# extract_all_backbones()


### 3.1 Detector features (Model D)

Nine values per frame, extracted in Stage 1: `{confidence, presence_flag,
count}` for each of `hand`, `needle_driver`, `forceps`. Frames with no
detections give all-zero vectors, which is itself informative (absence of
instruments in frame, not missing data). Bovie is excluded — 8 detections
across 140 sampled Stage-1 frames made it unusable as a feature and
unevaluable as a class.

This cell only *loads* Stage 1's cache — it does not re-run detection.


In [ ]:
def load_detector_features(video_id: str) -> pd.DataFrame:
    """Expects DIRS['detector']/{video_id}.csv with columns:
    frame_idx, {cls}_confidence, {cls}_present, {cls}_count for cls in
    CFG.detector_classes. Missing rows (no detection run for that frame)
    are filled with zeros — itself informative, per the spec."""
    path = DIRS["detector"] / f"{video_id}.csv"
    cols = ["frame_idx"] + [
        f"{c}_{f}" for c in CFG.detector_classes for f in ("confidence", "present", "count")
    ]
    if not path.exists():
        warnings.warn(f"No Stage 1 detector cache for {video_id} at {path}; "
                       "returning all zeros. Model D results using this video are unreliable.")
        manifest = pd.read_csv(DIRS["manifest"] / "frames_manifest.csv")
        frame_indices = manifest[manifest.video_id == video_id].frame_idx.tolist()
        df = pd.DataFrame(0.0, index=range(len(frame_indices)), columns=cols)
        df["frame_idx"] = frame_indices
        return df
    df = pd.read_csv(path)
    return df.reindex(columns=cols, fill_value=0.0)


DETECTOR_FEATURE_COLS = [
    f"{c}_{f}" for c in CFG.detector_classes for f in ("confidence", "present", "count")
]
print(f"Model D adds {len(DETECTOR_FEATURE_COLS)} detector dims: {DETECTOR_FEATURE_COLS}")


## 4. t-SNE per backbone, coloured by phase

A minutes-long sanity check, not a metric. If phases show no separation for
a given backbone here, the GRU will struggle regardless of how much
temporal modelling is layered on top — this is a quick early warning, and
its plots feed directly into Stage 3.


In [ ]:
def plot_tsne_per_backbone(max_points_per_video: int = 800):
    manifest = pd.read_csv(DIRS["manifest"] / "frames_manifest.csv")
    palette = sns.color_palette("tab10", N_CLASSES)

    for backbone_name in CFG.backbones:
        feats, phases = [], []
        for video_id in CFG.videos:
            f = load_cached_features(backbone_name, video_id)
            vid_rows = manifest[manifest.video_id == video_id].sort_values("frame_idx")
            idx = np.arange(len(f))
            if len(idx) > max_points_per_video:
                idx = np.random.choice(idx, max_points_per_video, replace=False)
            feats.append(f[idx])
            phases.append(vid_rows.phase.values[idx])
        X = np.concatenate(feats)
        y = np.concatenate(phases)

        emb = TSNE(n_components=2, init="pca", random_state=CFG.seed, perplexity=30).fit_transform(X)

        plt.figure(figsize=(7, 6))
        for i, phase in enumerate(CFG.phase_taxonomy):
            mask = y == phase
            if mask.any():
                plt.scatter(emb[mask, 0], emb[mask, 1], s=6, color=palette[i], label=phase, alpha=0.6)
        plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
        plt.title(f"t-SNE — {backbone_name}")
        plt.tight_layout()
        out_path = DIRS["tsne"] / f"tsne_{backbone_name}.png"
        plt.savefig(out_path, dpi=150)
        plt.show()
        print(f"Saved {out_path}")


# plot_tsne_per_backbone()


## 5. Leave-one-video-out cross-validation

Each fold trains a fresh model from scratch on six videos and predicts on
the held-out one; nothing carries between folds. Splitting is **by video**,
never by frame — adjacent frames are near-identical, so a random frame
split would let the model see frame 500 in training and frame 501 in
testing, producing excellent scores while learning nothing that
generalises to an unseen operation. Each fold uses a fixed seed derived
from `CFG.seed` for reproducibility.


In [ ]:
def loocv_folds():
    for fold_idx, held_out in enumerate(CFG.videos):
        train_videos = [v for v in CFG.videos if v != held_out]
        yield fold_idx, held_out, train_videos


def fold_seed(fold_idx: int) -> int:
    return CFG.seed + fold_idx


def compute_class_weights(manifest: pd.DataFrame, train_videos: list) -> torch.Tensor:
    """Inverse class frequency, computed per fold from training videos only."""
    train_manifest = manifest[manifest.video_id.isin(train_videos)]
    counts = train_manifest.phase_idx.value_counts().reindex(range(N_CLASSES), fill_value=0)
    freq = counts.values.astype(np.float64)
    freq[freq == 0] = 1.0  # phase absent from this fold's training set entirely
    weights = freq.sum() / (N_CLASSES * freq)
    return torch.tensor(weights, dtype=torch.float32)


## 6. Baselines

### 6.1 Baseline 1 — majority class

Always predicts the longest phase (by training-fold frame count). Cheap and
non-negotiable: without it, no other number in this notebook can be
interpreted as "genuine recognition" versus "exploiting duration imbalance."


In [ ]:
def majority_class_baseline(manifest: pd.DataFrame):
    all_preds, all_true, all_video = [], [], []
    per_video_rows = []

    for fold_idx, held_out, train_videos in loocv_folds():
        train_manifest = manifest[manifest.video_id.isin(train_videos)]
        majority_phase_idx = train_manifest.phase_idx.value_counts().idxmax()

        test_manifest = manifest[manifest.video_id == held_out]
        preds = np.full(len(test_manifest), majority_phase_idx)
        true = test_manifest.phase_idx.values

        all_preds.append(preds); all_true.append(true)
        all_video += [held_out] * len(test_manifest)

        per_video_rows.append({
            "video_id": held_out,
            "macro_f1": f1_score(true, preds, labels=range(N_CLASSES), average="macro", zero_division=0),
        })

    pooled_preds = np.concatenate(all_preds)
    pooled_true = np.concatenate(all_true)
    return {
        "name": "majority_class",
        "pooled_preds": pooled_preds,
        "pooled_true": pooled_true,
        "pooled_video": np.array(all_video),
        "per_video": pd.DataFrame(per_video_rows),
    }


# majority_result = majority_class_baseline(manifest)


### 6.2 Baseline 2 — per-frame classifier

Logistic regression on frozen features (winning backbone once selected;
before that, run per backbone alongside the sweep), with no temporal model
at all. If the GRU does not beat this, the temporal component is not
earning its place. Scaler and classifier are both fit per fold, on training
videos only.


In [ ]:
def per_frame_baseline(manifest: pd.DataFrame, backbone_name: str, model_type: str = "logreg"):
    all_preds, all_true, all_video = [], [], []
    per_video_rows = []

    feat_cache = {vid: load_cached_features(backbone_name, vid) for vid in CFG.videos}

    for fold_idx, held_out, train_videos in loocv_folds():
        set_seed(fold_seed(fold_idx))
        train_manifest = manifest[manifest.video_id.isin(train_videos)]
        test_manifest = manifest[manifest.video_id == held_out]

        # Cached features are stored in the same frame order as the manifest, so
        # slicing feat_cache[v][:len(vm)] lines feature rows up with manifest rows.
        X_train_parts, y_train_parts = [], []
        for v in train_videos:
            vm = train_manifest[train_manifest.video_id == v].sort_values("frame_idx")
            X_train_parts.append(feat_cache[v][: len(vm)])
            y_train_parts.append(vm.phase_idx.values)
        X_train = np.concatenate(X_train_parts)
        y_train = np.concatenate(y_train_parts)

        vm_test = test_manifest.sort_values("frame_idx")
        X_test = feat_cache[held_out][: len(vm_test)]
        y_test = vm_test.phase_idx.values

        scaler = StandardScaler().fit(X_train)
        X_train_s, X_test_s = scaler.transform(X_train), scaler.transform(X_test)

        class_weight = compute_class_weights(manifest, train_videos).numpy()
        sample_weight = class_weight[y_train]

        if model_type == "logreg":
            clf = LogisticRegression(max_iter=2000, multi_class="multinomial")
            clf.fit(X_train_s, y_train, sample_weight=sample_weight)
        else:
            clf = MLPClassifier(hidden_layer_sizes=(256,), max_iter=200, random_state=fold_seed(fold_idx))
            clf.fit(X_train_s, y_train)  # MLPClassifier has no sample_weight support

        preds = clf.predict(X_test_s)

        all_preds.append(preds); all_true.append(y_test)
        all_video += [held_out] * len(y_test)
        per_video_rows.append({
            "video_id": held_out,
            "macro_f1": f1_score(y_test, preds, labels=range(N_CLASSES), average="macro", zero_division=0),
        })

    pooled_preds = np.concatenate(all_preds)
    pooled_true = np.concatenate(all_true)
    return {
        "name": f"per_frame_{model_type}_{backbone_name}",
        "pooled_preds": pooled_preds,
        "pooled_true": pooled_true,
        "pooled_video": np.array(all_video),
        "per_video": pd.DataFrame(per_video_rows),
    }


# per_frame_result = per_frame_baseline(manifest, backbone_name="dinov2_vitb14")


## 7. BiGRU phase-recognition model

Frozen backbone features -> linear projection -> single-layer bidirectional
GRU -> per-frame classification head. Output is a phase prediction for
**every frame** in the sequence, not one label per clip.

Bidirectionality is appropriate here because this is offline retrospective
analysis — unlike an intraoperative system, there's no requirement to
predict without future context. A single GRU layer is deliberate: with six
training videos per fold, more depth overfits before it contributes
anything.

### Sequence handling

Videos run 11,000+ frames at 2 fps — too long for one forward pass.
Sequences are cut into overlapping windows of `CFG.window_size` (1024,
~8.5 minutes of real time) with `CFG.window_stride` (512) between window
starts, giving the model enough context to observe phase transitions.
At inference, logits from overlapping windows are averaged per frame
before taking the argmax.


In [ ]:
class PhaseGRU(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int = 256, n_classes: int = N_CLASSES):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.ReLU())
        self.gru = nn.GRU(hidden_dim, hidden_dim, num_layers=1, batch_first=True, bidirectional=True)
        self.head = nn.Linear(hidden_dim * 2, n_classes)

    def forward(self, x):
        # x: (batch, seq_len, input_dim)
        x = self.proj(x)
        x, _ = self.gru(x)
        return self.head(x)  # (batch, seq_len, n_classes)


def make_windows(seq_len: int, window_size: int, stride: int):
    """Start indices of overlapping windows covering [0, seq_len)."""
    if seq_len <= window_size:
        return [0]
    starts = list(range(0, seq_len - window_size + 1, stride))
    if starts[-1] + window_size < seq_len:
        starts.append(seq_len - window_size)
    return starts


class WindowedSequenceDataset(Dataset):
    """One item per (video, window). Windows are pre-computed so a batch
    can mix windows from different training videos."""

    def __init__(self, features_by_video: dict, labels_by_video: dict, window_size: int, stride: int):
        self.window_size = window_size
        self.items = []  # (video_id, start)
        self.features_by_video = features_by_video
        self.labels_by_video = labels_by_video
        for video_id, feats in features_by_video.items():
            for start in make_windows(len(feats), window_size, stride):
                self.items.append((video_id, start))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        video_id, start = self.items[i]
        feats = self.features_by_video[video_id]
        labels = self.labels_by_video[video_id]
        end = min(start + self.window_size, len(feats))
        x = feats[start:end]
        y = labels[start:end]
        if len(x) < self.window_size:  # pad final short window
            pad = self.window_size - len(x)
            x = np.pad(x, ((0, pad), (0, 0)))
            y = np.pad(y, (0, pad), constant_values=-100)  # ignore_index
        return torch.from_numpy(x).float(), torch.from_numpy(y).long()


In [ ]:
def train_phase_gru(features_by_video: dict, labels_by_video: dict, class_weights: torch.Tensor,
                     seed: int, input_dim: int, epochs: int = 25, lr: float = 1e-3, batch_size: int = 4):
    set_seed(seed)
    dataset = WindowedSequenceDataset(features_by_video, labels_by_video, CFG.window_size, CFG.window_stride)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=False)

    model = PhaseGRU(input_dim=input_dim).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE), ignore_index=-100)

    model.train()
    for epoch in range(epochs):
        total_loss = 0.0
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            logits = model(x)  # (B, T, C)
            loss = criterion(logits.reshape(-1, N_CLASSES), y.reshape(-1))
            loss.backward()
            opt.step()
            total_loss += loss.item() * x.size(0)
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"  epoch {epoch + 1}/{epochs}  loss={total_loss / len(dataset):.4f}")
    return model


@torch.no_grad()
def predict_phase_gru(model: PhaseGRU, feats: np.ndarray) -> np.ndarray:
    """Full-sequence logits via overlapping windows, averaged per frame."""
    model.eval()
    seq_len = len(feats)
    logit_sum = np.zeros((seq_len, N_CLASSES), dtype=np.float64)
    count = np.zeros(seq_len, dtype=np.int32)

    for start in make_windows(seq_len, CFG.window_size, CFG.window_stride):
        end = min(start + CFG.window_size, seq_len)
        x = feats[start:end]
        pad = CFG.window_size - len(x)
        if pad > 0:
            x = np.pad(x, ((0, pad), (0, 0)))
        x_t = torch.from_numpy(x).float().unsqueeze(0).to(DEVICE)
        logits = model(x_t)[0].cpu().numpy()[: end - start]
        logit_sum[start:end] += logits
        count[start:end] += 1

    avg_logits = logit_sum / count[:, None]
    return avg_logits  # (seq_len, N_CLASSES) — argmax externally, raw logits saved for Section 11


### 7.1 Feature standardisation (per fold)

Model C uses backbone features alone; Model D concatenates standardised
backbone features with the nine standardised detector values. **The two
feature blocks are standardised separately before concatenation, scalers
fit per fold** — otherwise nine detector dimensions sitting next to 768 or
2048 backbone dimensions would be numerically swamped regardless of their
information content.


In [ ]:
def build_fold_arrays(manifest: pd.DataFrame, backbone_name: str, train_videos: list, held_out: str,
                       use_detector: bool = False):
    backbone_cache = {vid: load_cached_features(backbone_name, vid) for vid in CFG.videos}
    detector_cache = None
    if use_detector:
        detector_cache = {
            vid: load_detector_features(vid)[DETECTOR_FEATURE_COLS].values.astype(np.float32)
            for vid in CFG.videos
        }

    backbone_scaler = StandardScaler().fit(np.concatenate([backbone_cache[v] for v in train_videos]))
    detector_scaler = None
    if use_detector:
        detector_scaler = StandardScaler().fit(np.concatenate([detector_cache[v] for v in train_videos]))

    def build_video(video_id):
        vm = manifest[manifest.video_id == video_id].sort_values("frame_idx")
        n = len(vm)
        feats = backbone_scaler.transform(backbone_cache[video_id][:n])
        if use_detector:
            det = detector_scaler.transform(detector_cache[video_id][:n])
            feats = np.concatenate([feats, det], axis=1)
        return feats.astype(np.float32), vm.phase_idx.values.astype(np.int64)

    features_by_video, labels_by_video = {}, {}
    for v in train_videos + [held_out]:
        features_by_video[v], labels_by_video[v] = build_video(v)

    input_dim = features_by_video[held_out].shape[1]
    return features_by_video, labels_by_video, input_dim


## 8. Metrics

### 8.1 Pooled metrics

All seven folds' held-out predictions are concatenated, then metrics
computed once — pooling avoids the instability of per-video macro-F1, which
would average over a different phase set in each video depending on which
phases that particular surgery happened to contain.

- **Macro-F1** is the headline, not accuracy: accuracy would be dominated
  by `urethroplasty` and `glansplasty` and would look strong while the
  model ignored everything brief.
- **Per-phase F1** is reported in full — with this taxonomy the per-phase
  table is likely more informative than any single summary number.
- **Confusion matrix**, pooled across folds — adjacent-phase confusion is a
  different, more forgivable failure than random confusion, and only the
  matrix shows which is happening.


In [ ]:
def pooled_metrics(pooled_true: np.ndarray, pooled_preds: np.ndarray, name: str = ""):
    macro_f1 = f1_score(pooled_true, pooled_preds, labels=range(N_CLASSES), average="macro", zero_division=0)
    per_phase_f1 = f1_score(pooled_true, pooled_preds, labels=range(N_CLASSES), average=None, zero_division=0)
    per_phase_f1 = pd.Series(per_phase_f1, index=CFG.phase_taxonomy, name="f1")

    cm = confusion_matrix(pooled_true, pooled_preds, labels=range(N_CLASSES))
    cm_df = pd.DataFrame(cm, index=CFG.phase_taxonomy, columns=CFG.phase_taxonomy)

    print(f"=== {name}: pooled macro-F1 = {macro_f1:.4f} ===")
    display(per_phase_f1.to_frame())

    plt.figure(figsize=(8, 7))
    sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues")
    plt.title(f"Pooled confusion matrix — {name}")
    plt.ylabel("True phase"); plt.xlabel("Predicted phase")
    plt.tight_layout()
    plt.savefig(DIRS["results"] / f"confusion_matrix_{name}.png", dpi=150)
    plt.show()

    return {"macro_f1": macro_f1, "per_phase_f1": per_phase_f1, "confusion_matrix": cm_df}


### 8.2 Segmental metrics

Frame-wise metrics reward a model that's right on average but flickers
between phases moment to moment. That fragmentation directly damages
Stage 4, where segment selection assumes coherent phase boundaries — so it
has to be measured here, not assumed. Edit score (Levenshtein distance
between predicted and true phase-segment sequences, normalised) and
segmental F1 at IoU thresholds follow the standard action-segmentation
formulation (Lea et al., Farha & Gall).


In [ ]:
def frames_to_segments(labels: np.ndarray):
    """Collapse a per-frame label sequence into (label, start, end) runs."""
    segments = []
    start = 0
    for i in range(1, len(labels) + 1):
        if i == len(labels) or labels[i] != labels[start]:
            segments.append((labels[start], start, i - 1))
            start = i
    return segments


def levenshtein(a, b):
    dp = np.zeros((len(a) + 1, len(b) + 1), dtype=int)
    dp[:, 0] = np.arange(len(a) + 1)
    dp[0, :] = np.arange(len(b) + 1)
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            dp[i, j] = min(dp[i - 1, j] + 1, dp[i, j - 1] + 1, dp[i - 1, j - 1] + cost)
    return dp[-1, -1]


def edit_score(true_labels: np.ndarray, pred_labels: np.ndarray) -> float:
    true_seq = [s[0] for s in frames_to_segments(true_labels)]
    pred_seq = [s[0] for s in frames_to_segments(pred_labels)]
    if len(true_seq) == 0 and len(pred_seq) == 0:
        return 100.0
    dist = levenshtein(true_seq, pred_seq)
    return (1 - dist / max(len(true_seq), len(pred_seq))) * 100


def segmental_f1_at_iou(true_labels: np.ndarray, pred_labels: np.ndarray, iou_threshold: float) -> float:
    """Segmental F1@IoU, matched greedily by descending IoU within each class
    (Farha & Gall, 2019 / Lea et al., 2016 formulation)."""
    true_segs = frames_to_segments(true_labels)
    pred_segs = frames_to_segments(pred_labels)

    def iou(seg_a, seg_b):
        _, sa, ea = seg_a; _, sb, eb = seg_b
        inter = max(0, min(ea, eb) - max(sa, sb) + 1)
        union = max(ea, eb) - min(sa, sb) + 1
        return inter / union if union > 0 else 0.0

    matched_true = set()
    tp = 0
    for p_seg in pred_segs:
        best_iou, best_j = 0.0, -1
        for j, t_seg in enumerate(true_segs):
            if j in matched_true or t_seg[0] != p_seg[0]:
                continue
            cur_iou = iou(p_seg, t_seg)
            if cur_iou > best_iou:
                best_iou, best_j = cur_iou, j
        if best_iou >= iou_threshold and best_j >= 0:
            tp += 1
            matched_true.add(best_j)

    fp = len(pred_segs) - tp
    fn = len(true_segs) - tp
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall) * 100


def segmental_metrics_per_video(pooled_true, pooled_preds, pooled_video):
    rows = []
    for video_id in CFG.videos:
        mask = pooled_video == video_id
        t, p = pooled_true[mask], pooled_preds[mask]
        rows.append({
            "video_id": video_id,
            "edit_score": edit_score(t, p),
            "f1@10": segmental_f1_at_iou(t, p, 0.10),
            "f1@25": segmental_f1_at_iou(t, p, 0.25),
            "f1@50": segmental_f1_at_iou(t, p, 0.50),
        })
    return pd.DataFrame(rows)


### 8.3 Per-video metrics

Computed in addition to the pooled numbers, to support the paired
comparison (Section 9) and populate the per-video results table. Noisier
than pooled metrics, and subject to the same phase-set caveat: a video only
containing a subset of phases will only ever contribute to that subset's
per-phase scores.


In [ ]:
def per_video_macro_f1(pooled_true, pooled_preds, pooled_video):
    rows = []
    for video_id in CFG.videos:
        mask = pooled_video == video_id
        rows.append({
            "video_id": video_id,
            "macro_f1": f1_score(pooled_true[mask], pooled_preds[mask], labels=range(N_CLASSES),
                                  average="macro", zero_division=0),
        })
    return pd.DataFrame(rows)


## 9. Running the comparison arms

`run_gru_loocv` trains and evaluates one full LOOCV sweep (7 folds) for a
given backbone, optionally with detector features concatenated in (Model
D). It also saves, per fold:

- raw per-video logits (Section 11), so metrics can be recomputed later
  without retraining;
- **the trained model's weights**, named after the video that fold held
  out (`{run_name}__{video_id}.pt`), plus a small JSON recording exactly
  which six videos trained that checkpoint. This is what Stage 4 needs and
  Section 11's logits alone can't provide: gradient-based saliency
  requires backpropagating through the actual model, not reading back a
  saved prediction. The naming convention *is* the video → model lookup —
  there is no separate mapping to keep in sync — and `load_fold_checkpoint`
  below refuses to load a checkpoint for a video that was in its own
  training set, so a mismatch fails loudly instead of silently leaking a
  video into its own highlight reel.


In [ ]:
FOLD_CKPT_DIR = DIRS["checkpoints"] / "gru_folds"
FOLD_CKPT_DIR.mkdir(parents=True, exist_ok=True)


def run_gru_loocv(manifest: pd.DataFrame, backbone_name: str, use_detector: bool = False,
                   run_name: str = None, epochs: int = 25, hidden_dim: int = 256):
    run_name = run_name or (f"{backbone_name}{'_plus_detector' if use_detector else ''}_gru")
    all_true, all_preds, all_video = [], [], []
    per_video_rows = []

    for fold_idx, held_out, train_videos in loocv_folds():
        print(f"--- {run_name} | fold {fold_idx + 1}/{CFG.n_folds} | held out {held_out} ---")
        features_by_video, labels_by_video, input_dim = build_fold_arrays(
            manifest, backbone_name, train_videos, held_out, use_detector=use_detector
        )
        class_weights = compute_class_weights(manifest, train_videos)

        train_feats = {v: features_by_video[v] for v in train_videos}
        train_labels = {v: labels_by_video[v] for v in train_videos}
        model = train_phase_gru(train_feats, train_labels, class_weights,
                                 seed=fold_seed(fold_idx), input_dim=input_dim, epochs=epochs)

        logits = predict_phase_gru(model, features_by_video[held_out])
        preds = logits.argmax(axis=1)
        true = labels_by_video[held_out]

        np.save(DIRS["logits"] / f"{run_name}__{held_out}.npy", logits)

        # Checkpoint the exact model that held this video out — the only
        # model Stage 4 may legitimately use to build this video's reel.
        ckpt_stem = f"{run_name}__{held_out}"
        torch.save(model.state_dict(), FOLD_CKPT_DIR / f"{ckpt_stem}.pt")
        with open(FOLD_CKPT_DIR / f"{ckpt_stem}.json", "w") as f:
            json.dump({
                "run_name": run_name,
                "held_out_video": held_out,
                "train_videos": train_videos,
                "backbone": backbone_name,
                "use_detector": use_detector,
                "input_dim": input_dim,
                "hidden_dim": hidden_dim,
                "n_classes": N_CLASSES,
                "fold_seed": fold_seed(fold_idx),
            }, f, indent=2)

        all_true.append(true); all_preds.append(preds); all_video += [held_out] * len(true)
        per_video_rows.append({
            "video_id": held_out,
            "macro_f1": f1_score(true, preds, labels=range(N_CLASSES), average="macro", zero_division=0),
        })

    return {
        "name": run_name,
        "pooled_true": np.concatenate(all_true),
        "pooled_preds": np.concatenate(all_preds),
        "pooled_video": np.array(all_video),
        "per_video": pd.DataFrame(per_video_rows),
    }


def load_fold_checkpoint(run_name: str, video_id: str) -> "tuple[PhaseGRU, dict]":
    """Load the one model legitimately usable to build `video_id`'s highlight
    reel: the fold of `run_name` that held `video_id` out. Refuses to load
    (raises) if the checkpoint's own metadata shows `video_id` was actually
    in that fold's training set — a wrong-file mix-up fails loudly here
    instead of silently leaking a video into its own reel later."""
    ckpt_stem = f"{run_name}__{video_id}"
    meta_path = FOLD_CKPT_DIR / f"{ckpt_stem}.json"
    weights_path = FOLD_CKPT_DIR / f"{ckpt_stem}.pt"
    if not meta_path.exists() or not weights_path.exists():
        raise FileNotFoundError(
            f"No checkpoint for run={run_name!r}, held-out video={video_id!r} at {weights_path}. "
            "Did run_gru_loocv finish for this run?"
        )

    with open(meta_path) as f:
        meta = json.load(f)

    if meta["held_out_video"] != video_id or video_id in meta["train_videos"]:
        raise AssertionError(
            f"Checkpoint metadata mismatch for {weights_path}: expected held-out video "
            f"{video_id!r}, but metadata says held_out_video={meta['held_out_video']!r}, "
            f"train_videos={meta['train_videos']}. Refusing to load — using this checkpoint "
            f"would let {video_id!r} leak into its own highlight reel."
        )

    model = PhaseGRU(input_dim=meta["input_dim"], hidden_dim=meta["hidden_dim"], n_classes=meta["n_classes"])
    model.load_state_dict(torch.load(weights_path, map_location=DEVICE))
    model.to(DEVICE).eval()
    return model, meta


### 9.1 Backbone sweep

All four backbones under identical conditions — same frames, same
hyperparameters, same folds, same seeds. Only the input features differ.
Reported descriptively (no significance testing across backbones); the
best pooled macro-F1 carries forward as the "winning backbone" for Model C
and Model D.


In [ ]:
def run_backbone_sweep(manifest: pd.DataFrame, epochs: int = 25):
    sweep_results = {}
    for backbone_name in CFG.backbones:
        sweep_results[backbone_name] = run_gru_loocv(manifest, backbone_name, use_detector=False, epochs=epochs)

    summary = pd.DataFrame([
        {"backbone": name, "pooled_macro_f1": f1_score(
            r["pooled_true"], r["pooled_preds"], labels=range(N_CLASSES), average="macro", zero_division=0)}
        for name, r in sweep_results.items()
    ]).sort_values("pooled_macro_f1", ascending=False)
    display(summary)
    summary.to_csv(DIRS["results"] / "backbone_sweep_summary.csv", index=False)

    winning_backbone = summary.iloc[0].backbone
    print(f"Winning backbone (carries forward as Model C/D input): {winning_backbone}")
    return sweep_results, winning_backbone


# sweep_results, winning_backbone = run_backbone_sweep(manifest)


### 9.2 Model C vs Model D

**Model C** — winning backbone -> GRU (the reference system).
**Model D** — winning backbone + detector features -> GRU. This is the
pre-specified question: does making instrument presence explicit improve
phase recognition beyond what self-supervised visual features already
encode implicitly?

**Conditionality on Stage 1.** Model D depends on the Stage 1 zero-shot
detector. If Stage 1's detection F1 is poor, Model D below should be
reported as *a test with unreliable inputs*, not as evidence about whether
instrument information helps — a null result would otherwise be
uninterpretable, since it could mean either "instrument information
doesn't help" or "the detector was too weak to supply it." Check the Stage
1 detection F1 before drawing any conclusion from a negative Model D
result, and if Model D shows no benefit, first verify the per-block
standardisation in Section 7.1 rather than concluding the detector signal
is genuinely unhelpful.


In [ ]:
STAGE1_DETECTION_F1 = None  # fill in from Stage 1's notebook output before interpreting Model D
STAGE1_F1_RELIABILITY_THRESHOLD = 0.5  # edit if Stage 1 defines its own bar

def check_model_d_conditionality():
    if STAGE1_DETECTION_F1 is None:
        print("STAGE1_DETECTION_F1 is unset — set it from Stage 1's output before trusting Model D's result.")
    elif STAGE1_DETECTION_F1 < STAGE1_F1_RELIABILITY_THRESHOLD:
        print(f"Stage 1 detection F1 = {STAGE1_DETECTION_F1:.3f} is below the reliability threshold "
              f"({STAGE1_F1_RELIABILITY_THRESHOLD}). Model D must be reported as a test with unreliable "
              "inputs, not as evidence about whether instrument information aids phase recognition.")
    else:
        print(f"Stage 1 detection F1 = {STAGE1_DETECTION_F1:.3f} — Model D is interpretable as intended.")


# check_model_d_conditionality()

def run_model_c_and_d(manifest: pd.DataFrame, winning_backbone: str, epochs: int = 25):
    model_c = run_gru_loocv(manifest, winning_backbone, use_detector=False, run_name="model_C", epochs=epochs)
    model_d = run_gru_loocv(manifest, winning_backbone, use_detector=True, run_name="model_D", epochs=epochs)
    return model_c, model_d


# model_c, model_d = run_model_c_and_d(manifest, winning_backbone)


## 10. Statistics

**Feasibility framing.** At n=7, the exact minimum two-sided p-value for a
paired Wilcoxon signed-rank test is 0.0156, achievable only on a unanimous
seven-video sweep. That does not survive Bonferroni correction across more
than three comparisons, and is mathematically unreachable at six videos.
This bounds what any inferential claim below can honestly say.

- Full per-video table — seven rows, all arms. At this sample size the raw
  data can simply be shown, which is more informative than any summary
  statistic.
- Win counts — "Model D outperformed Model C on k of 7 videos."
- Cliff's delta with bootstrap confidence intervals, reported as a
  descriptive effect size.
- **One pre-specified test: Model C vs Model D**, Wilcoxon signed-rank,
  uncorrected, declared before running as the sole inferential comparison.
  The backbone sweep and both baselines are reported descriptively,
  without testing.
- The backbone was selected using the same data as the C vs D comparison —
  a mild optimism bias, stated here explicitly rather than corrected for.
- Non-significant results are reported as inconclusive, never as
  equivalence.
- LOOCV training sets overlap almost completely (six of seven videos
  shared between any two folds), so paired tests violate the independence
  assumption the framework assumes. Inferential results below are
  described as indicative, with emphasis on effect sizes and the pooled
  confusion matrix rather than the p-value alone.


In [ ]:
def cliffs_delta(a: np.ndarray, b: np.ndarray) -> float:
    a, b = np.asarray(a), np.asarray(b)
    gt = sum(x > y for x in a for y in b)
    lt = sum(x < y for x in a for y in b)
    return (gt - lt) / (len(a) * len(b))


def bootstrap_cliffs_delta_ci(a: np.ndarray, b: np.ndarray, n_boot: int = 5000, alpha: float = 0.05, seed: int = 0):
    rng = np.random.default_rng(seed)
    deltas = []
    for _ in range(n_boot):
        a_bs = rng.choice(a, size=len(a), replace=True)
        b_bs = rng.choice(b, size=len(b), replace=True)
        deltas.append(cliffs_delta(a_bs, b_bs))
    lo, hi = np.percentile(deltas, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return lo, hi


def compare_arms(results_by_name: dict, arm_a: str, arm_b: str, run_test: bool = False):
    """Builds the seven-row paired per-video table for two arms, win counts,
    Cliff's delta + bootstrap CI, and (only when run_test=True — reserved
    for the single pre-specified Model C vs Model D comparison) the
    Wilcoxon signed-rank test."""
    a = results_by_name[arm_a]["per_video"].set_index("video_id")["macro_f1"]
    b = results_by_name[arm_b]["per_video"].set_index("video_id")["macro_f1"]
    table = pd.DataFrame({arm_a: a, arm_b: b})
    table["diff"] = table[arm_b] - table[arm_a]
    display(table)

    wins_b = int((table["diff"] > 0).sum())
    ties = int((table["diff"] == 0).sum())
    print(f"{arm_b} outperformed {arm_a} on {wins_b} of {len(table)} videos ({ties} ties).")

    delta = cliffs_delta(table[arm_b].values, table[arm_a].values)
    ci_lo, ci_hi = bootstrap_cliffs_delta_ci(table[arm_b].values, table[arm_a].values, seed=CFG.seed)
    print(f"Cliff's delta ({arm_b} vs {arm_a}): {delta:.3f}  [95% bootstrap CI {ci_lo:.3f}, {ci_hi:.3f}] (descriptive)")

    result = {"table": table, "wins_b": wins_b, "ties": ties, "cliffs_delta": delta, "cliffs_delta_ci": (ci_lo, ci_hi)}

    if run_test:
        stat, p = wilcoxon(table[arm_a].values, table[arm_b].values)
        print(f"\nPre-specified test — Wilcoxon signed-rank, {arm_a} vs {arm_b}: statistic={stat:.3f}, p={p:.4f}")
        print("Minimum reachable two-sided p at n=7 is 0.0156 (unanimous sweep). "
              "Uncorrected, single pre-specified comparison. "
              "A non-significant result here is reported as inconclusive, not as equivalence. "
              "LOOCV folds share 6/7 training videos pairwise, so treat this as indicative rather than "
              "a clean independence-assumption test — weight the effect size and pooled confusion matrix "
              "at least as heavily as this p-value.")
        result["wilcoxon_statistic"] = stat
        result["wilcoxon_p"] = p

    return result


# c_vs_d = compare_arms(
#     {"model_C": model_c, "model_D": model_d}, "model_C", "model_D", run_test=True
# )


## 11. Outputs

Raw per-video logits are already saved to `DIRS['logits']` by
`run_gru_loocv` as each fold finishes (`{run_name}__{video_id}.npy`), so
every metric above can be recomputed without retraining. The trained model
weights for that same fold are saved alongside them under
`FOLD_CKPT_DIR` (`DIRS['checkpoints']/gru_folds/{run_name}__{video_id}.pt`
+ a matching `.json` recording that fold's training-video list) — this is
what Stage 4 loads via `load_fold_checkpoint(run_name, video_id)` to
compute gradient-based saliency for that specific video, using the one
model that never saw it during training. This section writes the
remaining artefacts: per-video results tables, pooled metrics, confusion
matrices (already saved as PNGs in Section 8), and a run manifest
recording the configuration and code version behind this run.


In [ ]:
def save_run_metadata(extra: dict = None):
    try:
        git_hash = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=str(ROOT)).decode().strip()
    except Exception:
        git_hash = "unknown (not a git checkout, or git unavailable)"

    meta = {
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
        "git_hash": git_hash,
        "config": {
            "videos": CFG.videos,
            "phase_taxonomy": CFG.phase_taxonomy,
            "gap_policy": CFG.gap_policy,
            "base_fps": CFG.base_fps,
            "min_frames_per_segment": CFG.min_frames_per_segment,
            "window_size": CFG.window_size,
            "window_stride": CFG.window_stride,
            "seed": CFG.seed,
            "backbones": CFG.backbones,
            "detector_classes": CFG.detector_classes,
        },
    }
    if extra:
        meta.update(extra)

    out_path = DIRS["results"] / f"run_metadata_{meta['timestamp'].replace(':', '-')}.json"
    with open(out_path, "w") as f:
        json.dump(meta, f, indent=2)
    print(f"Saved run metadata: {out_path}")
    return meta


def save_results_tables(results_by_name: dict):
    for name, result in results_by_name.items():
        result["per_video"].to_csv(DIRS["results"] / f"per_video_macro_f1_{name}.csv", index=False)
        seg = segmental_metrics_per_video(result["pooled_true"], result["pooled_preds"], result["pooled_video"])
        seg.to_csv(DIRS["results"] / f"segmental_metrics_{name}.csv", index=False)
    print(f"Saved per-video and segmental results tables for: {list(results_by_name)}")


# save_run_metadata()
# save_results_tables({"majority_class": majority_result, "model_C": model_c, "model_D": model_d})


## 12. Putting it together

Uncomment and run in order once `CFG` points at real data (raw videos in
`DIRS['videos']`, CVAT exports in `DIRS['annotations']`, and — for Model D
— the Stage 1 detector cache in `DIRS['detector']`).


In [ ]:
# --- 1. Manifest ---
# manifest = build_full_manifest()
# incidence, frame_counts, mode_counts = build_incidence_artefacts(manifest)

# --- 2. Feature extraction ---
# extract_all_backbones()

# --- 3. t-SNE sanity check ---
# plot_tsne_per_backbone()

# --- 4 & 5. Baselines ---
# majority_result = majority_class_baseline(manifest)
# pooled_metrics(majority_result["pooled_true"], majority_result["pooled_preds"], name="majority_class")

# --- 6. Backbone sweep (includes the per-frame baseline on the winning backbone) ---
# sweep_results, winning_backbone = run_backbone_sweep(manifest)
# per_frame_result = per_frame_baseline(manifest, backbone_name=winning_backbone)
# pooled_metrics(per_frame_result["pooled_true"], per_frame_result["pooled_preds"], name="per_frame_baseline")

# --- 7. Model C vs Model D ---
# check_model_d_conditionality()
# model_c, model_d = run_model_c_and_d(manifest, winning_backbone)
# pooled_metrics(model_c["pooled_true"], model_c["pooled_preds"], name="model_C")
# pooled_metrics(model_d["pooled_true"], model_d["pooled_preds"], name="model_D")
# c_vs_d = compare_arms({"model_C": model_c, "model_D": model_d}, "model_C", "model_D", run_test=True)

# --- Outputs ---
# save_run_metadata()
# save_results_tables({
#     "majority_class": majority_result, "per_frame_baseline": per_frame_result,
#     "model_C": model_c, "model_D": model_d,
# })
